# 17 — BeanBox Coffee Orders: Two-Source Take-Home (Cleaning + Join + Gold + Unit Test + Delta Recovery)

**Scenario.** BeanBox is a small Dutch online shop for coffee capsules and accessories. Finance rejected last quarter's revenue report: the totals looked wrong, and they changed every time the analyst re-ran the notebook. I got the raw order export (Parquet) and the product catalog (CSV) as-is, with no documentation. The job: make the data trustworthy, build a small reporting layer, and be able to defend every number.

Compared to the earlier projects, three things are new here:
- **Two source formats**: Parquet keeps its types, CSV arrives as all strings. Same profiling ritual for both.
- **Every gold question twice**: DataFrame API and Spark SQL, and the two results must match. That is my reconciliation.
- **Part 2 engineering steps**: the G1 logic extracted into a tested function, the join strategy read from the physical plan, and a deliberate bad write repaired with Delta `RESTORE`.

Timebox for Part 1 was 60 minutes, solo. Part 2 was done on a second day.

## 1. Bronze — read the files as they are

The orders export is Parquet, so it already carries types (`qty` came in as double, `order_date` as string). I keep whatever the file says at this stage; nothing is cast before it is measured.

In [ ]:
orders_17_bronze_df = (
    spark.read.format('parquet')
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/orders_17.parquet")

)
orders_17_bronze_df.limit(5).display()

In [ ]:
orders_17_bronze_df.printSchema()

In [ ]:
print(orders_17_bronze_df.count())
print(len(orders_17_bronze_df.columns))

## 2. Profiling orders — grain first

The grain I expect is **one row = one order**. Before trusting that, count vs distinct count on `order_id`, then look at the duplicated rows with my own eyes: are they exact copies or different orders sharing an id?

In [ ]:
from pyspark.sql.functions import col

print(orders_17_bronze_df.select("order_id").count())
print(orders_17_bronze_df.select("order_id").distinct().count())

orders_17_bronze_df.groupBy(col("order_id")).count().filter(col("count")>1).display()

In [ ]:
orders_17_bronze_df.filter(col("order_id").isin("ORD-5089","ORD-5006","ORD-5031")).display()

**Measured:** 107 rows, 104 distinct `order_id`. The three repeated ids (ORD-5006, ORD-5031, ORD-5089) are identical in every column, so a full-row `dropDuplicates()` is the right tool. No key-based dedup: that would silently pick one of two different orders.

In [ ]:
orders_17_dedup_df = orders_17_bronze_df.dropDuplicates()
orders_17_dedup_df.count()

In [ ]:
print(orders_17_dedup_df.filter(col("order_id").isNull()).count())
print(orders_17_dedup_df.filter(col("order_date").isNull()).count())
print(orders_17_dedup_df.filter(col("customer_id").isNull()).count())
print(orders_17_dedup_df.filter(col("product_id").isNull()).count())
print(orders_17_dedup_df.filter(col("qty").isNull()).count())
print(orders_17_dedup_df.filter(col("unit_price").isNull()).count())
print(orders_17_dedup_df.filter(col("status").isNull()).count())

In [ ]:
orders_17_dedup_df.groupBy("qty").count().display()
orders_17_dedup_df.groupBy("unit_price").count().display()
orders_17_dedup_df.groupBy("status").count().display()

**Measured after dedup (104 rows):**
- `qty`: 1 null and 1 negative value (−1, on a delivered order)
- `unit_price`: string with currency dirt (`€`, comma decimals) plus 3 junk values (`error`, `unknown`, `n/a`)
- `status`: 16 raw spellings (case variants, leading spaces, one `unknown`) that should be 4 values
- `order_date`: string in two formats, `d/M/yyyy` and `yyyy-M-d`

## 3. Cleaning

### 3.1 Normalize strings

`trim` on every string column, `lower` on `status`. Only string columns: trimming a number makes no sense, numbers get a range check instead.

In [ ]:
from pyspark.sql.functions import trim, initcap,lower,col

orders_silver_df = orders_17_dedup_df.withColumns({
    "order_id"         :  trim(col("order_id")),
    "order_date"       :  trim(col("order_date")),
    "customer_id"      :  trim(col("customer_id")),
    "product_id"       :  trim(col("product_id")),
    "unit_price"       :  trim(col("unit_price")),
    "status"           :  trim(lower(col("status")))
})

orders_silver_df.limit(5).display()

### 3.2 Currency dirt before the cast

`€12,50` cannot become a decimal as it is. Remove the symbol, turn the comma into a dot, *then* cast. Cleaning after the cast is impossible: the cast would already have produced nulls.

In [ ]:
from pyspark.sql.functions import replace,lit

orders_silver_df = orders_silver_df.withColumns({
    "unit_price" : replace(replace(col("unit_price"),lit("€"),lit("")),lit(","),lit("."))
})

orders_silver_df.limit(5).display()

### 3.3 Placeholders → NULL

Systematic sweep over every column: which ones still contain `error` / `unknown` / `n/a`? Then replace with null in the two columns that have them. A placeholder is not a value; leaving it in would crash the cast (ANSI mode) or corrupt counts.

In [ ]:
print(orders_silver_df.filter(lower(col("order_id")).isin("error","n/a","unknown")).count())
print(orders_silver_df.filter(lower(col("order_date")).isin("error","n/a","unknown")).count())
print(orders_silver_df.filter(lower(col("customer_id")).isin("error","n/a","unknown")).count())
print(orders_silver_df.filter(lower(col("product_id")).isin("error","n/a","unknown")).count())
print(orders_silver_df.filter(lower(col("qty")).isin("error","n/a","unknown")).count())
print(orders_silver_df.filter(lower(col("unit_price")).isin("error","n/a","unknown")).count())
print(orders_silver_df.filter(lower(col("status")).isin("error","n/a","unknown")).count())

In [ ]:
from pyspark.sql.functions import when 

orders_silver_df = orders_silver_df.withColumns({
    "unit_price"  : when(lower(col("unit_price")).isin("unknown","n/a","error"),None).otherwise(col("unit_price")),
    "status"      : when(lower(col("status")).isin("unknown","n/a","error"),None).otherwise(col("status"))
})

print(orders_silver_df.filter(lower(col("qty")).isin("unknown","error","n/a")).count())
print(orders_silver_df.filter(lower(col("unit_price")).isin("unknown","error","n/a")).count())

In [ ]:
orders_silver_df.filter(col("unit_price").isNull()).count()

In [ ]:
orders_silver_df.groupBy("status").count().display()

**Measured:** 3 junk prices → null, 1 unknown status → null. Status is now `delivered` 78 · `cancelled` 12 · `shipped` 9 · `returned` 4 · null 1.

### 3.4 Types — clean first, cast after, count nulls on both sides

- `order_date`: `coalesce` of two `try_to_date` attempts, one per format.
- `qty`: int (a quantity is counted, not measured).
- `unit_price`: `decimal(10,2)`, never float for money.

The proof that nothing was lost silently: the number of nulls *after* the cast must equal the number of junk values *before* it. 3 in, 3 out.

In [ ]:
from pyspark.sql.functions import cast, coalesce, try_to_date


orders_silver_df = orders_silver_df.withColumns({
    "order_date"   :   coalesce(
        try_to_date("order_date", "d/M/yyyy"),
        try_to_date("order_date","yyyy-M-d")

    ),
    "qty"          :  col("qty").cast("int"),
    "unit_price"   :  col("unit_price").cast("decimal(10,2)")
})

orders_silver_df.printSchema()

In [ ]:
orders_silver_df.filter((col("qty")<0) | (col("unit_price")<=0)).display()

**Business repair, a decision rather than a step.** One delivered order has `qty = -1`. Three options: set it to 0, delete the row, set it to null. I set it to **null and keep the row**. Deleting would remove a real delivered order from revenue; 0 would turn "unknown" into a real number. Null says "quantity unknown" and finance can still see the order.

In [ ]:
orders_silver_df = orders_silver_df.withColumn("qty", when(col("qty")<0,None).otherwise(col("qty")))

In [ ]:
orders_silver_df.filter(col("qty").isNull()).count()

In [ ]:
from pyspark.sql.functions import min,max

orders_silver_df.select(min("order_date"), max("order_date")).display()

In [ ]:
print(orders_silver_df.count())
print(orders_silver_df.filter(col("order_date").isNull()).count())

**Measured:** 104 rows, 0 unparsed dates, date range 2025-01-01 → 2025-06-30 (H1, as expected). Nulls now: `qty` 2, `unit_price` 3, `status` 1.

One more number the gold questions will need: how many **delivered** orders have no usable price? These stay in the table but contribute nothing to revenue, and finance has to know that.

In [ ]:
orders_silver_df.filter((col("status") == "delivered") & col("unit_price").isNull()).display()
print(orders_silver_df.filter((col("status") == "delivered") & col("unit_price").isNull()).count())

**Measured:** 2 delivered orders without a price (ORD-5004, ORD-5067). Flagged, not fixed: I cannot invent a price.

## 4. Silver — idempotent write + Delta proofs

`overwrite` is the whole answer to finance's complaint: running the notebook five times produces the same 104 rows, not 520. After the write, two cheap checks that I do on every project: `DESCRIBE DETAIL` (a photo: how many files right now) and `DESCRIBE HISTORY` (a diary: who wrote what, when).

In [ ]:
orders_silver_df.write.mode("overwrite").saveAsTable("orders_silver_17")

In [ ]:
%sql
DESCRIBE DETAIL orders_silver_17

In [ ]:
%sql
describe history orders_silver_17

**Reading the history:** every run is one `CREATE OR REPLACE TABLE AS SELECT` version. In `operationMetrics`, each version writes exactly **104 rows and 3,320 bytes** and removes the previous **3,320 bytes**. Same input, same output, run after run. That is what idempotent looks like in a log, and it is the number I would show finance. `numFiles = 1` because 104 rows fit in one partition; `numRemovedFiles` is 0 on version 0 and 1 afterwards (Delta marks the old file as removed, it does not delete it yet).

### Products — same ritual, smaller table

12 rows is not a reason to skip profiling: the same table could have 12,000 rows next quarter.

In [ ]:
products_bronze_17_df = (
    spark.read.format("csv")
    .option("header", True)
    .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/products_17.csv")
)

products_bronze_17_df.limit(5).display()

In [ ]:
products_bronze_17_df.count()

In [ ]:
print(products_bronze_17_df.select("product_id").count())
print(products_bronze_17_df.select("product_id").distinct().count())

In [ ]:
print(products_bronze_17_df.filter(col("category").isNull()).count())
print(products_bronze_17_df.filter(col("product_id").isNull()).count())
print(products_bronze_17_df.filter(col("product_name").isNull()).count())
print(products_bronze_17_df.filter(col("list_price_eur").isNull()).count())

In [ ]:
print(products_bronze_17_df.filter(lower(col("category")).isin("unknown","error","n/a")).count())
print(products_bronze_17_df.filter(lower(col("product_id")).isin("unknown","error","n/a")).count())
print(products_bronze_17_df.filter(lower(col("product_name")).isin("unknown","error","n/a")).count())
print(products_bronze_17_df.filter(lower(col("list_price_eur")).isin("unknown","error","n/a")).count())

**Measured:** 12 rows, 12 distinct `product_id`, 0 nulls, 0 placeholders. Clean lookup. Only cosmetic work: `initcap` on names/categories, and `list_price_eur` renamed to `price` and cast to decimal.

In [ ]:
products_silver_17_df = products_bronze_17_df.withColumns({
    "category"       : trim(initcap(col("category"))),
    "product_id"     : trim(col("product_id")),
    "product_name"   : trim(initcap(col("product_name"))),
    "price"          : trim(col("list_price_eur"))
}).drop("list_price_eur")

products_silver_17_df.limit(5).display()

In [ ]:
products_silver_17_df = products_silver_17_df.withColumns({
    "price"     :  col("price").cast("decimal(10,2)")
})

products_silver_17_df.printSchema()

In [ ]:
products_silver_17_df.write.mode("overwrite").saveAsTable("products_silver_17")

## 5. Join — orphans first, then join type, then fan-out proof

Before joining I ask both directions: orders whose product is not in the catalog, and catalog products with no orders (that second one is G3). The first direction decides the join type.

In [ ]:
%sql
with joined as (
select 
o.product_id as op,
p.product_id as pp,
o.qty,
o.status,
p.product_name,
p.category,
o.unit_price
from orders_silver_17 as o left join products_silver_17 as p 
on p.product_id = o.product_id)

select * from joined
where pp is null

**Measured:** 2 orders reference **P-999**, a product that does not exist in the catalog (one delivered at €5.20, one shipped). Those are real orders. An **inner join would silently drop them**; a **left join** keeps them with a null category, visible in G1 as their own line, and the catalog team gets the question. Join type is a business decision, not syntax.

The products side is 12 rows, so I hint `broadcast()` and read the plan afterwards (section 9).

In [ ]:
from pyspark.sql.functions import broadcast

orders_enriched_df = orders_silver_df.join(broadcast(products_silver_17_df), how='left', on='product_id')
orders_enriched_df.limit(5).display()

In [ ]:
print(orders_silver_df.count())
print(orders_enriched_df.count())

**Grain statement:** one row in `orders_enriched` = **one order**, enriched with product name, category and list price. Proven: 104 rows before the join, 104 after. No fan-out, because `product_id` is unique in products (checked in profiling, 12 = 12).

## 6. Gold — every question in SQL and in the DataFrame API

Two independent paths to the same number. If they disagree, one of them has a bug; if they agree, I trust the number.

### G1 — Delivered revenue per category, H1 2025

In [ ]:
%sql
with delivered_orders as (
select 
o.order_date,
o.product_id as op,
p.product_id as pp,
o.qty,
o.status,
p.product_name,
p.category,
o.unit_price
from orders_silver_17 as o left join products_silver_17 as p 
on p.product_id = o.product_id
where o.status = 'delivered'),

y as (
select
order_date, 
qty*unit_price as revenue,
category
from delivered_orders
where order_date < '2025-07-01'
order by revenue desc)

select 
category,
sum(revenue) as total_revenue
from y 
group by category
order by total_revenue desc

In [ ]:
delivered_orders_df = (orders_enriched_df
    .withColumn("revenue", col("qty") * col("unit_price"))
    .filter(col("status") == "delivered"))

revenue_per_category_df = (delivered_orders_df
    .groupBy("category")
    .agg(expr("sum(qty * unit_price) as total_revenue"))
    .orderBy(col("total_revenue").desc()))

revenue_per_category_df.display()

**Answer (delivered, EUR), SQL and API identical:**

| category | revenue |
|---|---|
| Accessories | 930.70 |
| Espresso | 228.50 |
| Lungo | 128.60 |
| Decaf | 86.10 |
| *(no category, P-999)* | 5.20 |

Total delivered revenue: **€1,379.10** across 76 priced delivered orders. Accessories are two thirds of H1 revenue, the capsule categories together are under a third. Two delivered orders have no price and are not in these numbers, and the P-999 line is a catalog gap, not a category; both go back to the owners as questions. Confidence: high on the totals (two paths agree, 104 = 104), medium on the category split until P-999 is resolved.

### G2 [W] — Monthly revenue with a 3-month moving average

The window frame is the whole point of this question. `avg(...) over (order by month)` without a frame silently gives a **running** average, because the default frame is "everything up to the current row". `rows between 2 preceding and current row` turns it into the moving average finance asked for. Nothing errors either way; only the numbers change.

In [ ]:
orders_enriched_df.createOrReplaceTempView("orders_enriched")

In [ ]:
%sql
with monthly_revenue as (
select
date_format(order_date,'yyyy-MM') as order_month, 
sum(qty * unit_price) as monthly_revenue
from orders_enriched 
where status = 'delivered'
group by order_month
order by order_month)

select * , 
avg(monthly_revenue) over(order by order_month ROWS BETWEEN 2 PRECEDING AND CURRENT ROW) as moving_avg_3months,
sum(monthly_revenue) OVER (ORDER BY order_month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total
from monthly_revenue

In [ ]:
from pyspark.sql.functions import date_format, sum

monthly_revenue_df = (delivered_orders_df
    .withColumn("order_month", date_format("order_date", "yyyy-MM"))
    .groupBy("order_month")
    .agg(sum("revenue").alias("monthly_revenue"))
    .orderBy("order_month"))
monthly_revenue_df.display()

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

w_3m = Window.orderBy("order_month").rowsBetween(-2, 0)

monthly_revenue_df = monthly_revenue_df.withColumn("moving_avg_3m", avg("monthly_revenue").over(w_3m))
monthly_revenue_df.display()

**Answer (delivered, EUR), SQL and API identical:**

| month | revenue | 3-month moving avg |
|---|---|---|
| 2025-01 | 264.70 | 264.70 |
| 2025-02 | 240.70 | 252.70 |
| 2025-03 | 224.40 | 243.27 |
| 2025-04 | 144.95 | 203.35 |
| 2025-05 | 187.95 | 185.77 |
| 2025-06 | 316.40 | 216.43 |

Checked by hand: April = (224.40 + 144.95 + 187.95) / 3 = 203.35. The running total in the SQL version ends at 1,379.10, which is exactly the G1 total, so the two questions reconcile. Revenue dipped through April and recovered in June, the strongest month of the half; with six data points I would call that a pattern to watch, not a trend.

### G3 — Catalog products with zero sales in H1

The mirror of the orphan check: products on the left, orders on the right, keep what has no partner. SQL says `left join ... where order_id is null`; the API has a name for it, `left_anti`.

In [ ]:
%sql
select
  p.product_id,
  p.product_name
from products_silver_17 as p
left join orders_silver_17 as o
  on p.product_id = o.product_id
where o.order_id is null

In [ ]:
products_silver_17_df.join(orders_silver_df, how="left_anti", on="product_id").display()

**Answer:** two products never sold in H1: **P-206 Lungo Bio** and **P-212 Travel Mug**. Not to be confused with P-999, which is the opposite direction (sold, but missing from the catalog).

## 7. Gold tables written idempotently + reconciliation

Each answer becomes its own Delta table, named after the question, written with `overwrite`. Then one last cross-check: the sum of the category table must equal the sum of the monthly table.

In [ ]:
revenue_per_category_df.write.mode("overwrite").saveAsTable("gold_revenue_per_category_17")
monthly_revenue_df.write.mode("overwrite").saveAsTable("gold_monthly_revenue_17")
(products_silver_17_df.join(orders_silver_df, how="left_anti", on="product_id")
    .write.mode("overwrite").saveAsTable("gold_products_never_sold_17"))

In [ ]:
from pyspark.sql.functions import round as spark_round

by_category = spark.table("gold_revenue_per_category_17").agg(sum("total_revenue")).collect()[0][0]
by_month = spark.table("gold_monthly_revenue_17").agg(sum("monthly_revenue")).collect()[0][0]
print(by_category, by_month)
assert by_category == by_month, "category total and monthly total disagree" 

## 8. Unit test on the G1 logic

The G1 aggregation moved into a plain function in `beanbox_functions.py`. The three extraction rules: it **returns** a DataFrame, it takes the DataFrame as a **parameter** (the name inside the function is just a slot; it does not have to match the name outside), and it knows nothing about file paths or tables.

```python
from pyspark.sql.functions import expr, col

def revenue_per_category(df):
    result = df.groupBy("category").agg(
        expr("sum(qty * unit_price) as total_revenue")
    ).orderBy(col("total_revenue").desc())
    return result
```

The test in `test_beanbox_functions.py` feeds six hand-made rows (two per category) and compares against **hand-written** expected totals, 150 / 60 / 90. Two things I learned while writing it:
- **Spark infers the type from the Python value.** `50` is an int, `50.0` a double; a decimal only appears through an explicit cast or a `Decimal` object. My first expected schema said `decimal(10,2)` and failed on `150` (not a `Decimal`); the second attempt failed on `LongType` vs `DoubleType` because the input was still int. Test data in doubles, done.
- **A green test proves the logic, not the data.** It says the function sums per category correctly. It says nothing about whether the real 104 rows are clean; that is what the profiling and reconciliation above are for.

I also broke the test on purpose (expected 999 instead of 150) to see the red: `[DIFFERENT_ROWS] 33.33 %`, the differing row marked with `!`, and the notebook's `assert retcode == 0` gate firing. Green means nothing until you have seen red once.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pytest
import sys
sys.dont_write_bytecode = True

retcode = pytest.main(["-v", "test_beanbox_functions.py"])
assert retcode == 0, "unit tests failed" 

## 9. Performance — which join strategy did Spark pick, and why is it right?

The plan is the proof, not the hint.

In [ ]:
orders_enriched_df.explain()

The relevant line:

```
PhotonBroadcastHashJoin [product_id#...], [product_id#...], LeftOuter, BuildRight, ...
```

- `BroadcastHashJoin` + `BuildRight`: the **right** side (products, 12 rows) is copied to every executor; the orders side stays where it is.
- `LeftOuter`: my left-join decision, visible in the plan.
- The only `Exchange` in the whole plan comes from the full-row dedup, not from the join.

In one sentence: Spark broadcast the 12-row products table to every executor, so the orders table never had to be shuffled; copying a tiny table is cheaper than moving the big one. At 104 rows this changes nothing; at 100 million rows it is the difference between a join that runs and one that spends its time on the network. The hint was not even needed: products is far under the 10 MB `autoBroadcastJoinThreshold`, Spark would have chosen the same strategy on its own.

## 10. Delta recovery — break the silver table, then bring it back

The scenario finance fears: a job or a colleague writes bad data into silver. Delta keeps every version in its log, so this is a rehearsal, not a disaster. First the accident, on purpose.

In [ ]:
%sql
update orders_silver_17 set unit_price = unit_price * 100 
where status = 'delivered'

78 rows changed (every delivered order). Now the damage, read **from the table**, not from the DataFrame variable: `orders_silver_df` is a plan that recomputes from bronze and would still show the old prices. To see what is on disk, read the table.

In [ ]:
spark.table("orders_silver_17").filter(col("status") == "delivered").limit(3).display()

In [ ]:
%sql
describe history orders_silver_17

Two things in the history:
- **Version 5 = UPDATE**, with the predicate stored: `status = delivered`. The log knows exactly which rows were touched.
- **Version 6 = OPTIMIZE, `auto: true`**. I never ran it. Databricks compacted the files by itself two seconds after my write.

That is the trap when restoring: "one version back" is the OPTIMIZE, still holding the bad prices; two back is the UPDATE itself. The last version I trust is **4**, the last clean overwrite. Read the history, then choose.

In [ ]:
%sql
restore table orders_silver_17 to version as of 4

In [ ]:
%sql
select * from orders_silver_17
where status = 'delivered'
limit 3

**Restore output:** 1 file removed (3,383 bytes, the compacted bad one), 1 file restored (3,320 bytes, exactly the size of my clean write), about ten seconds. Prices are back to 5.20 / 4.70. The history now has a version 7, `RESTORE`, because a restore is itself a write; nothing is erased from the log.

Delta never deleted the old file, it only marked it as removed. That is why the rollback costs nothing until `VACUUM` runs, and why the default 7-day retention is not something to lower just to save 3 KB: the retention window is what makes `RESTORE` possible.

In one sentence for the interview: *if a run writes bad data, I do not need a backup; I read `DESCRIBE HISTORY`, find the last version I trust, and restore it.*

## 11. Decision log

| # | decision | evidence / number | alternative cost |
|---|---|---|---|
| 1 | Dropped 3 full-row duplicates only | 107 → 104; ORD-5006/5031/5089 identical in every column; count = distinct after | key-based dedup could delete a real second order |
| 2 | Negative qty → **null**, row kept | ORD-5059, delivered, qty −1 | deleting loses a delivered order; 0 invents a number |
| 3 | Junk prices → null, then cast, counted on both sides | 3 junk in, 3 nulls out | casting first would hide the loss as "just nulls" |
| 4 | **Left join** + null category for orphans | P-999 in 2 orders, one delivered (€5.20) | inner join drops real sales silently |
| 5 | Unpriced delivered orders **kept and flagged**, not estimated | 2 orders (ORD-5004, ORD-5067) | an estimated price would be my number in finance's report |
| 6 | Moving average with an explicit `rows between 2 preceding and current row` frame | April 203.35 checked by hand | the default frame gives a running average, no error |
| 7 | Every gold answer in SQL and API, totals reconciled | G1 total 1,379.10 = G2 running total; 104 = 104 | a single path cannot tell you it is wrong |
| 8 | `overwrite` everywhere | five runs, 104 rows and 3,320 bytes each time in the log | append would double revenue on every re-run, finance's original complaint |
| 9 | Restore to version 4, not 6 or 5 | v5 = UPDATE, v6 = auto OPTIMIZE | "one version back" would have kept the bad prices |

## 12. Defense questions

**D1. Your revenue total depends on how you treated unpriced and unknown-status orders. What did you decide, and how would you explain it to finance?**
Unpriced delivered orders (2) are in the silver table but contribute nothing to revenue, because I will not invent a price. The unknown status (1) became null and is excluded from "delivered". To finance I would say: the €1,379.10 is a floor, it covers 76 priced delivered orders; two more delivered orders exist without a price and here are their ids. They decide whether to chase the price or write them off.

**D2. Finance said the old report changed on every re-run. What in your pipeline guarantees that yours does not?**
Every table is written with `overwrite`, so a re-run replaces the table instead of adding to it. The proof is in `DESCRIBE HISTORY`: five runs, each writing exactly 104 rows and 3,320 bytes. Same input, same output. If the old report used `append`, every run added the same rows again and the total grew, which is exactly what they observed.

**D3. Two orders reference a product that is not in the catalog. What did you do with them, and what is the risk of the alternative?**
I kept them with a left join; they show in G1 as a line with no category (€5.20 delivered). The alternative, an inner join, would have dropped them silently: no error, just €5.20 less revenue and nobody knowing. Small here, but the mechanism does not care about size.

## 13. Key takeaways

- **Profile before you cast, and count nulls on both sides of every cast.** 3 in, 3 out is a proof; "some nulls" is a guess.
- **Delete only what is provably not information** (exact copies). Everything else: null the value, keep the row.
- **Join type is a business decision.** Ask "which rows have no partner?" before choosing, then prove the row count did not change.
- **Write every gold answer twice.** SQL and API agreeing is the cheapest reconciliation there is.
- **A window without a frame is not wrong, it is a different question.** Check one value by hand.
- **The Delta log is evidence.** Idempotency, who changed what, and the way back are all in `DESCRIBE HISTORY`.
- **A DataFrame is a plan, a table is a snapshot.** To see what is on disk, read the table.
- **Restore to the last version you trust, not the last version.** The platform adds versions you did not write.